# Phase 24 — RA-MedFiLM architecture specification

Reconstructed notebook for `phase_24_ra_medfilm_design`. This phase is **design-only**: it froze the module contract that Phase 25 later trained. Nothing here is an executed experiment.

> **Status:** reconstructed analysis notebook. It reads persisted artifacts only. No training, no new inference, no protected TEST access.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, r"D:\Softfors\jup\full_doc_report\Scripts")
import remedfilm_guard as GUARD
GUARD.install()

REPO = Path(r"D:\Softfors\jup")
REPORT = REPO / "full_doc_report"
TABLES = REPORT / "tables"
FIG = REPORT / "figures"

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

def fig(subdir, name, width=760):
    """Display a generated figure, failing loudly if it is missing."""
    p = FIG / subdir / f"{name}.png"
    if not p.exists():
        raise FileNotFoundError(f"missing figure: {p}")
    display(Image(filename=str(p), width=width))

def table(path, **kw):
    return pd.read_csv(path, **kw)

GUARD.require()

True

In [2]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified


In [3]:
spec = pd.read_csv(REPO / "reports/tables/phase24_module_spec.csv")
spec

,module_id,name,symbol,role,novelty_bearing,parameter_target,gt_free_at_inference,executed_in_phase24,notes
0,GLOBAL,Frozen Global MedFiLM reference,G,fallback output + reusable features,no,"3523001 (frozen, measured in Phase 21)",yes,no,Phase-21 SHA bound; MUST NOT be modified in in...
1,ROI,GT-free region proposal,P_roi,propose refinement region,no,0 (rule-based),yes,no,foreground bbox + fixed context expansion; con...
2,R,Dedicated regional refiner,R_psi,produce optional refinement,no,"0.5M-1.5M recommended, <=2.0M hard ceiling (TA...",yes,no,crop-trained lightweight residual U-Net; NOT t...
3,B,Benefit router (paired-gain predictor),B_phi,predict paired refinement gain BEFORE refinement,YES (in integration),<=250k params; <=2% global MACs (preferred 1%)...,yes,no,small MLP / shallow feature fusion; no transfo...
4,GATE,Pre-execution resource-aware gate,U_hat > tau,EXECUTE/SKIP decision,YES (with B),0,yes,no,must not require running R_psi (C4)
5,S,Post-refinement safety selector,S_omega,preserve quality after refinement,no (closest to JFS),0,yes,no,does NOT save compute; CONTROLLED_CORE_EXTENSION
6,FUSE,Fusion rule,F,compose final mask,no,0,yes,no,"outside ROI global, inside ROI refined; no lea..."


In [4]:
print("novelty-bearing modules:",
      int(spec.novelty_bearing.astype(str).str.lower().eq("true").sum()))
print("modules executed in phase 24:",
      int(spec.executed_in_phase24.astype(str).str.lower().eq("true").sum()))
print()
print("Phase 24 is a specification phase: the parameter targets below were")
print("realised later in Phase 25 and are checked against the final contracts.")

novelty-bearing modules: 0
modules executed in phase 24: 0

Phase 24 is a specification phase: the parameter targets below were
realised later in Phase 25 and are checked against the final contracts.


In [5]:
# Router input contract — which features the router is allowed to see.
rc = pd.read_csv(REPO / "reports/tables/phase24_router_input_contract.csv")
print("router feature rows:", len(rc))
rc.head(20)

router feature rows: 31


,feature,source,available_pre_refinement,uses_GT,inference_available,primary_or_optional,allowed,reason
0,global_encoder_pooled_features,F_global (global forward),True,False,True,primary,True,reuses the already-computed global forward; no...
1,global_decoder_pooled_features,F_global (global forward),True,False,True,primary,True,same reuse argument
2,roi_pooled_global_features,F_global pooled over the proposed ROI,True,False,True,primary,True,ROI is GT-free; pooling adds negligible cost
3,probability_entropy_summaries,p_global,True,False,True,primary,True,computed from the predicted map only
4,probability_margin_summaries,p_global,True,False,True,primary,True,predicted-map statistic
5,probability_percentiles,p_global,True,False,True,optional,True,cheap predicted-map summary
6,predicted_foreground_fraction,p_global (thresholded),True,False,True,primary,True,"from the PREDICTION, not the label"
7,predicted_component_count,p_global (connected components),True,False,True,optional,True,predicted-map topology
8,predicted_largest_component_ratio,p_global,True,False,True,optional,True,predicted-map topology
9,predicted_bbox_width_height,p_global (ROI bbox),True,False,True,primary,True,geometry of the PREDICTED foreground


In [6]:
# Planned ablation family. `executed_in_phase24` is False for every arm:
# Phase 24 planned the ablation, Phase 28 executed it.
plan = pd.read_csv(REPO / "reports/tables/phase24_ablation_plan.csv")
print("planned arms:", len(plan))
print(plan.family.value_counts().to_string())
plan[["ablation_id", "name", "family", "executed_in_phase24"]]

planned arms: 13
family
core       9
router     3
refiner    1


,ablation_id,name,family,executed_in_phase24
0,A0,global-only,core,no
1,A1,always-refine,core,no
2,A2,random routing,core,no
3,A3,entropy routing,core,no
4,A4,margin routing,core,no
5,A5,predicted-ROI-size routing,core,no
6,A6,learned paired-gain router,core,no
7,A7,A6 + resource cost,core,no
8,A8,A7 + post-safety,core,no
9,R1,router without deep global features,router,no


In [7]:
# Cross-check: the realised parameter counts against the Phase 24 targets.
real = pd.read_csv(TABLES / "REMEDFILM_COMPONENT_PARAMETERS.csv")
real[["symbol", "model_class", "role", "parameters"]]

,symbol,model_class,role,parameters
0,G,GlobalMedFiLM,GLOBAL_SEGMENTER,3523001
1,R_psi,BARRRegionalRefinerV1,REGIONAL_REFINER,1276401
2,B_phi,BARRBenefitRouterV1,BENEFIT_ROUTER,36801
3,S_omega,BARRPostSafetyV1,POST_SAFETY,1217
4,FULL,assembled,PROPOSED_SYSTEM,4837420


## Provenance

- `reports/tables/phase24_module_spec.csv`
- `reports/tables/phase24_router_input_contract.csv`
- `reports/tables/phase24_ablation_plan.csv`
- `reports/audits/phase24_exact_implementation_spec.json`

In [8]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified
